# Breakout 4B · Sprint 4 · results from tax feed customs

Tax audited a random sample of the tax numbers that clear imports (VAT periods 2024-10 to 2025-09, completed before customs month 25). Customs controlled its random draw in months 25 to 36. **Does a tax adjustment point to customs findings?** The customs hit rate with and without a tax adjustment, and the lift.

The same steps as the sheet S4 Tax feeds customs of `4B_compute.xlsx`, where each line's control result comes from the raw Controls sheet; here the declaration files carry it (`random_draw`, `outcome`). Stuck? `4B_compute_worked.xlsx` shows every formula.

*Published customs datasets record an irregularity on 1 to 8 lines in 100; in this lab the random draw finds 3 in 100.*

In [ ]:
# Run this cell first (click it, then Shift + Enter). It finds the lab data.
import hashlib, os
from pathlib import Path
import numpy as np, pandas as pd

CANDIDATES = [os.environ.get('WORKSHOP_DATA', ''), 'data', '../data', '/workspace/data', '.']
DATA = next(Path(c) for c in CANDIDATES if c and (Path(c) / 'tariff.csv').exists())
print('Data folder:', DATA)

def _h(v, nd):
    return hashlib.sha256(f'{round(float(v), nd):.{nd}f}'.encode()).hexdigest()[:16]
def _hs(v):
    return hashlib.sha256(str(v).strip().lower().encode()).hexdigest()[:16]
def check(name, value, expected, nd=None):
    if callable(value):
        try:
            value = value()
        except Exception:
            value = ...
    try:
        ok = value is not ... and (_hs(value) == expected if nd is None else _h(value, nd) == expected)
    except Exception:
        ok = False
    shown = value if nd is None or value is ... else f'{value:,.{nd}f}'
    print(('✓ ' if ok else '✗ not yet: ') + name + ' = ' + str(shown))


In [ ]:
if not (DATA / 'tax_audits.csv').exists():
    print('tax_audits.csv is not in this release yet: this notebook runs when it is.')
else:
    audits = pd.read_csv(DATA / 'tax_audits.csv')
    cols = ['period_month', 'tin_customs', 'random_draw', 'outcome']
    lines = pd.concat([pd.read_csv(DATA / f, usecols=cols, low_memory=False) for f in ('declarations_h1.csv', 'declarations_h2.csv')])
    draw = lines[(lines.period_month >= 25) & (lines.random_draw == 1)].copy(); draw['found'] = (draw.outcome == 'finding').astype(int)
    rnd = audits[audits.selection_basis == 'random']   # only the random audits can measure
    print(len(rnd), 'random audits')

## YOUR TURN

In [ ]:
# YOUR TURN
adjusted = ...     # 1 per tax number with an adjustment in the random audits, 0 with none
rate_with = ...    # customs hit rate on random-draw lines of numbers with a tax adjustment
rate_without = ... # the same, numbers audited with none
lift = ...         # rate_with ÷ rate_without; and how many lines behind each rate?

In [ ]:
check('customs hit rate on the lines of numbers with a tax adjustment', lambda: rate_with, '44896291821a8010', 3)
check('customs hit rate on the lines of numbers audited with none', lambda: rate_without, 'b4c3a133ffec5dd7', 3)
check('the lift', lambda: lift, '6c3f9363fae05325', 2)